# Week 1: Claude decides, your application executes

Customer support agent — five demos. Model: `claude-sonnet-4-6`.

**Arc:** API is stateless → tools are requests → `stop_reason` runs the loop → failure is data → hard rules in code, judgement in the prompt.


## Setup

In [39]:
%pip install -q anthropic python-dotenv

import json
import os
import random
from pathlib import Path

from anthropic import Anthropic
from dotenv import load_dotenv

for candidate in [
    Path(".env"),
    Path("../.env"),
    Path("demo-ui/.env.local"),
]:
    if candidate.exists():
        load_dotenv(candidate)
        break
else:
    load_dotenv()

MODEL = "claude-sonnet-4-6"
client = Anthropic(api_key=os.environ["ANTHROPIC_API_KEY"])
print("Ready. Model:", MODEL)



[notice] A new release of pip is available: 25.3 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
Ready. Model: claude-sonnet-4-6


## Demo 1 — Losing state, then supplying it

**Note:** Claude forgets everything between requests. The application owns conversation state and must resend the full transcript. Also show `content` as a list, `stop_reason`, `usage`, and how `max_tokens` truncates mid-output.


**Note:** First call — point at `content` (a list), `stop_reason`, and `usage`. Not just the text.


In [40]:
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    system="You are a customer support assistant for an online electronics store.",
    messages=[{"role": "user", "content": "Where is my order?"}],
)

print([b.type for b in response.content])
print(response.stop_reason)
print(response.usage)
print(response.content[0].text)

['text']
end_turn
Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='global', input_tokens=25, output_tokens=88, output_tokens_details=None, server_tool_use=None, service_tier='standard')
I'd be happy to help you track your order! To look into this for you, I'll need a few details:

1. **Order number** - This can be found in your confirmation email
2. **Email address** or **account information** associated with the order

Could you please provide those details? Once I have them, I can help you find out the status of your order. 😊


**Note:** Same follow-up, no history. Claude has no memory of the earlier turn.


In [41]:
# WRONG: no history supplied
bad = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    messages=[{"role": "user", "content": "What was the first thing I asked you?"}],
)
print(bad.content[0].text)

You haven't asked me anything before this question - this is the first message you've sent me in our conversation. I don't have access to any previous conversations we may have had, as each conversation starts fresh for me.


**Note:** Application resends the full transcript. Context is an architecture problem, not a prompting problem.


In [42]:
# RIGHT: the application supplies the full transcript
messages = [
    {"role": "user", "content": "Where is my order?"},
    {"role": "assistant", "content": response.content},
    {"role": "user", "content": "What was the first thing I asked you?"},
]

good = client.messages.create(model=MODEL, max_tokens=1024, messages=messages)
print(good.content[0].text)

Your first message to me was **"Where is my order?"**

Is there anything else I can help you with?


**Note:** `max_tokens` is a hard ceiling. `stop_reason` becomes `max_tokens` — output can be cut mid-structure.


In [43]:
truncated = client.messages.create(
    model=MODEL,
    max_tokens=16,
    messages=[{"role": "user", "content": "Explain the refund process in detail."}],
)

print(truncated.stop_reason)  # max_tokens
print(truncated.content[0].text)

max_tokens
I'd be happy to explain a refund process, but I should note that


## Demo 2 — One tool, executed by hand

**Note:** Claude returns a `tool_use` request — it does not run your code. Your app executes the function and sends back a `tool_result` keyed by `tool_use_id`. The tool description is what Claude reads when deciding to call it.


In [44]:
CUSTOMERS = {
    "alice@example.com": {
        "customer_id": "CUST-001",
        "name": "Alice Nguyen",
        "tier": "gold",
        "verified": True,
    },
    "bob@example.com": {
        "customer_id": "CUST-002",
        "name": "Bob Okafor",
        "tier": "standard",
        "verified": False,
    },
}


def get_customer(email):
    c = CUSTOMERS.get(email.lower().strip())
    return {"found": False} if c is None else {"found": True, **c}


get_customer_tool = {
    "name": "get_customer",
    "description": (
        "Look up a customer account by email address. Returns customer ID, "
        "name, tier, and verification status. Use before any action that "
        "requires knowing who the customer is."
    ),
    "input_schema": {
        "type": "object",
        "properties": {"email": {"type": "string"}},
        "required": ["email"],
    },
}

SYSTEM = "You are a customer support assistant. Use the tools available to you."

**Note:** Watch `stop_reason: tool_use`. Extract `id`, `name`, `input` — nothing has executed yet.


In [45]:
messages = [
    {
        "role": "user",
        "content": "Hi, it's alice@example.com. Can you check my account?",
    }
]

response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    system=SYSTEM,
    tools=[get_customer_tool],
    messages=messages,
)

print(response.stop_reason)  # tool_use

tool_block = next(b for b in response.content if b.type == "tool_use")
print(tool_block.id)
print(tool_block.name)
print(tool_block.input)

tool_use
toolu_01NaoSCzprjPuABTAe5tr7iU
get_customer
{'email': 'alice@example.com'}


**Note:** Your Python runs the tool. Append assistant content, then `tool_result` with matching `tool_use_id`.


In [46]:
result = get_customer(**tool_block.input)
print(result)

messages.append({"role": "assistant", "content": response.content})
messages.append(
    {
        "role": "user",
        "content": [
            {
                "type": "tool_result",
                "tool_use_id": tool_block.id,
                "content": json.dumps(result),
            }
        ],
    }
)

final = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    system=SYSTEM,
    tools=[get_customer_tool],
    messages=messages,
)
print(final.stop_reason)
print(final.content[0].text)

{'found': True, 'customer_id': 'CUST-001', 'name': 'Alice Nguyen', 'tier': 'gold', 'verified': True}
end_turn
I found your account! Here's a summary:

- **Name:** Alice Nguyen
- **Customer ID:** CUST-001
- **Tier:** Gold 🌟
- **Verification Status:** Verified ✅

Everything looks good! Is there anything else I can help you with?


### Descriptions decide selection

**Note:** Vague descriptions cause wrong or missing tool picks. Expand descriptions with inputs, examples, and boundaries — that is the fix, not few-shot routing or a separate router.


In [63]:
# Two tools, minimal descriptions
vague_tools = [
    {
        "name": "get_customer",
        "description": "Retrieves customer information.",
        "input_schema": {
            "type": "object",
            "properties": {"email": {"type": "string"}},
            "required": ["email"],
        },
    },
    {
        "name": "lookup_order",
        "description": "Retrieves order details.",
        "input_schema": {
            "type": "object",
            "properties": {"order_id": {"type": "string"}},
            "required": ["order_id"],
        },
    },
]

**Note:** Vague descriptions — model may call nothing or the wrong tool. Print `stop_reason` and any text.


In [64]:
# One call with minimal descriptions
r = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    system=SYSTEM,
    tools=vague_tools,
    messages=[{"role": "user", "content": "check my order #12345"}],
)
print("stop_reason:", r.stop_reason)
print("tools:", [b.name for b in r.content if b.type == "tool_use"])
print("inputs:", [b.input for b in r.content if b.type == "tool_use"])
for b in r.content:
    if b.type == "text":
        print("text:", b.text)


stop_reason: end_turn
tools: []
inputs: []
text: I'd be happy to look up your order! I'll also need your email address to pull up your customer account alongside the order details.

Could you please provide your email address?


In [49]:
# Same tools, descriptions expanded with inputs, examples and boundaries
sharp_tools = [
    {
        "name": "get_customer",
        "description": (
            "Look up a customer ACCOUNT by email address. Input: an email like "
            "alice@example.com. Returns customer ID, name, tier, verification status. "
            "Use when you need to know who the customer is. Do NOT use for order "
            "queries: use lookup_order for anything identified by an order number."
        ),
        "input_schema": {
            "type": "object",
            "properties": {"email": {"type": "string"}},
            "required": ["email"],
        },
    },
    {
        "name": "lookup_order",
        "description": (
            "Look up a single ORDER by order ID. Input: an ID like ORD-123, with or "
            "without the ORD prefix. Returns status, total, items, days since delivery. "
            "Use whenever the request references an order number, tracking, delivery or "
            "a purchase. Do NOT use to identify the customer: use get_customer."
        ),
        "input_schema": {
            "type": "object",
            "properties": {"order_id": {"type": "string"}},
            "required": ["order_id"],
        },
    },
]

**Note:** Same prompt, sharper descriptions — expect `lookup_order` with an order id.


In [50]:
# Same request, expanded descriptions
r = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    system=SYSTEM,
    tools=sharp_tools,
    messages=[{"role": "user", "content": "check my order #12345"}],
)
print("stop_reason:", r.stop_reason)
print("tools:", [b.name for b in r.content if b.type == "tool_use"])
print("inputs:", [b.input for b in r.content if b.type == "tool_use"])
for b in r.content:
    if b.type == "text":
        print("text:", b.text)


stop_reason: tool_use
tools: ['lookup_order']
inputs: [{'order_id': 'ORD-12345'}]
text: Sure! Let me look up that order for you right away.


## Demo 3 — Four tools, one loop

**Note:** Branch on `stop_reason` (`end_turn` / `tool_use` / `max_tokens`), not string matching or turn count. Handle every `tool_use` block in a response (parallel tools). Multi-concern requests should be decomposed in one agent run.


In [51]:
ORDERS = {
    "ORD-123": {
        "order_id": "ORD-123",
        "customer_id": "CUST-001",
        "status": "delivered",
        "total": 149.99,
        "days_since_delivery": 6,
    },
    "ORD-456": {
        "order_id": "ORD-456",
        "customer_id": "CUST-002",
        "status": "delivered",
        "total": 899.00,
        "days_since_delivery": 41,
    },
}

REFUND_LOG = []


def lookup_order(order_id):
    o = ORDERS.get(order_id.upper().strip())
    return {"found": False} if o is None else {"found": True, **o}


def calculate_refund(order_id):
    o = ORDERS.get(order_id.upper().strip())
    if o is None:
        return {"eligible": False, "reason": "order_not_found"}
    if o["days_since_delivery"] > 30:
        return {
            "eligible": False,
            "reason": "outside_30_day_window",
            "days_since_delivery": o["days_since_delivery"],
        }
    return {"eligible": True, "refund_amount": o["total"]}


def process_refund(order_id, amount):
    REFUND_LOG.append({"order_id": order_id, "amount": amount})
    return {"success": True, "confirmation": f"REF-{len(REFUND_LOG):04d}"}


TOOL_FUNCTIONS = {
    "get_customer": get_customer,
    "lookup_order": lookup_order,
    "calculate_refund": calculate_refund,
    "process_refund": process_refund,
}

TOOLS = [
    get_customer_tool,
    {
        "name": "lookup_order",
        "description": (
            "Look up an order by order ID. Returns status, total, customer ID, "
            "and days since delivery. Use when the customer mentions an order."
        ),
        "input_schema": {
            "type": "object",
            "properties": {"order_id": {"type": "string"}},
            "required": ["order_id"],
        },
    },
    {
        "name": "calculate_refund",
        "description": (
            "Check whether an order is eligible for a refund and return the "
            "refund amount. Orders more than 30 days past delivery are not eligible."
        ),
        "input_schema": {
            "type": "object",
            "properties": {"order_id": {"type": "string"}},
            "required": ["order_id"],
        },
    },
    {
        "name": "process_refund",
        "description": (
            "Issue a refund for an eligible order. Call only after calculate_refund "
            "confirms eligibility. Returns a confirmation code."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string"},
                "amount": {"type": "number"},
            },
            "required": ["order_id", "amount"],
        },
    },
]

SYSTEM = (
    "You are a customer support assistant for an online electronics store. "
    "Use the tools available to you. Identify the customer, look up the order, "
    "check refund eligibility, and process the refund when allowed. "
    "When a message asks about more than one order or concern, handle each one."
)

**Note:** The loop is a switch on `stop_reason`. Iterate every `tool_use` block — taking only the first drops work.


In [52]:
def run_agent(user_message, system=SYSTEM, tools=TOOLS, max_turns=10):
    messages = [{"role": "user", "content": user_message}]

    for turn in range(max_turns):
        response = client.messages.create(
            model=MODEL,
            max_tokens=1024,
            system=system,
            tools=tools,
            messages=messages,
        )

        if response.stop_reason == "end_turn":
            return response, messages

        if response.stop_reason == "max_tokens":
            raise RuntimeError("Response truncated. Raise max_tokens.")

        messages.append({"role": "assistant", "content": response.content})

        results = []
        for block in response.content:
            if block.type != "tool_use":
                continue
            out = TOOL_FUNCTIONS[block.name](**block.input)
            results.append(
                {
                    "type": "tool_result",
                    "tool_use_id": block.id,
                    "content": json.dumps(out),
                }
            )

        messages.append({"role": "user", "content": results})

    raise RuntimeError("Hit max_turns without end_turn.")

**Note:** Multi-concern request — refund ORD-123 and status of ORD-456. Look for parallel `tool_use` blocks.


In [53]:
final, transcript = run_agent(
    "I'm alice@example.com. I want a refund for ORD-123, "
    "and can you also tell me where ORD-456 is?"
)

print(final.stop_reason)
print(final.content[0].text)
print("REFUND_LOG:", REFUND_LOG)


end_turn
Here's a full summary, Alice:

**Refund for ORD-123 ✅**
- Refund of **$149.99** has been successfully processed.
- Confirmation code: **REF-0001**
- Please allow a few business days for the amount to appear on your original payment method.

**Status of ORD-456 🔒**
- This order is not linked to your account, so I'm unable to provide information on it. If you believe this is an error, please double-check the order ID or contact us with additional details.

Is there anything else I can help you with?
REFUND_LOG: [{'order_id': 'ORD-123', 'amount': 149.99}]


**Note:** This transcript was assembled by the application and resent each turn.


In [54]:
for m in transcript:
    if isinstance(m["content"], str):
        print(f"{m['role']:>9}: {m['content'][:70]}")
    else:
        for b in m["content"]:
            t = b.type if hasattr(b, "type") else b["type"]
            name = getattr(b, "name", None) or (
                b.get("name") if isinstance(b, dict) else None
            )
            label = f"{t}:{name}" if name else t
            print(f"{m['role']:>9}: [{label}]")


     user: I'm alice@example.com. I want a refund for ORD-123, and can you also t
assistant: [text]
assistant: [tool_use:get_customer]
assistant: [tool_use:lookup_order]
assistant: [tool_use:lookup_order]
     user: [tool_result]
     user: [tool_result]
     user: [tool_result]
assistant: [text]
assistant: [tool_use:calculate_refund]
     user: [tool_result]
assistant: [text]
assistant: [tool_use:process_refund]
     user: [tool_result]


## Demo 4 — Four failures, four outcomes

**Note:** Return structured errors instead of raising. Four categories: transient → retry; validation → fix input then retry; business → explain, do not retry; permission → stop and escalate. Keep tool results lean — debug fields cost tokens on every later turn. `tool_choice` forces a schema shape.


In [55]:
def tool_error(category, retryable, message, **extra):
    return {
        "ok": False,
        "errorCategory": category,
        "isRetryable": retryable,
        "message": message,
        **extra,
    }


FAIL_MODE = None


def lookup_order_v2(order_id):
    if FAIL_MODE == "transient":
        return tool_error(
            "transient", True, "Order service temporarily unavailable."
        )
    if FAIL_MODE == "validation":
        return tool_error(
            "validation",
            True,
            "order_id must match ORD-NNN.",
            received=order_id,
        )
    if FAIL_MODE == "permission":
        return tool_error(
            "permission",
            False,
            "Order belongs to a different account. Verify identity or escalate.",
        )
    return lookup_order(order_id)


# business error: policy violation, not retryable, customer-facing explanation
def calculate_refund_v2(order_id):
    if FAIL_MODE == "business":
        return tool_error(
            "business",
            False,
            "Orders are refundable within 30 days of delivery. "
            "This order is outside the refund window.",
        )
    o = ORDERS.get(order_id.upper().strip())
    if o is None:
        return tool_error("validation", True, "Unknown order ID.")
    if o["days_since_delivery"] > 30:
        return tool_error(
            "business",
            False,
            "Orders are refundable within 30 days of delivery. "
            f"This order was delivered {o['days_since_delivery']} days ago.",
            days_since_delivery=o["days_since_delivery"],
        )
    return {"eligible": True, "refund_amount": o["total"]}


def process_refund_v2(order_id, amount):
    # Catch agents that skip calculate_refund under FAIL_MODE=business
    if FAIL_MODE == "business":
        return tool_error(
            "business",
            False,
            "Orders are refundable within 30 days of delivery. "
            "This order is outside the refund window.",
        )
    return process_refund(order_id, amount)


TOOL_FUNCTIONS["lookup_order"] = lookup_order_v2
TOOL_FUNCTIONS["calculate_refund"] = calculate_refund_v2
TOOL_FUNCTIONS["process_refund"] = process_refund_v2

**Note:** Walk all four: transient → retry; validation → fix then retry; business → explain, no retry; permission → escalate.


In [56]:
for mode in ["transient", "validation", "permission", "business"]:
    FAIL_MODE = mode
    print("=" * 60)
    print("FAIL_MODE:", mode)
    resp, _ = run_agent("I'm alice@example.com, refund ORD-123 please.")
    print(resp.stop_reason)
    print(resp.content[0].text)

FAIL_MODE = None

FAIL_MODE: transient
end_turn
I'm sorry, Alice — the order service is still temporarily unavailable, so I'm unable to look up order **ORD-123** right now. Here's what I'd suggest:

- **Try again in a few minutes** — this is a temporary issue and should resolve shortly.
- **Contact us again** and we'll get your refund processed as soon as the service is back up.

Apologies for the inconvenience! Is there anything else I can help you with in the meantime?
FAIL_MODE: validation
end_turn
Hi, Alice! I found your account. However, there was an issue looking up order **ORD-123** — the system says the order ID format is invalid (it expects a format like `ORD-NNN` with more digits).

Could you double-check the order ID? It may be something like **ORD-1234** or similar. Once you confirm the correct order ID, I'll get the refund process started right away!
FAIL_MODE: permission
end_turn
I was able to verify your identity, **Alice Nguyen**, but I'm unable to process the refund for **ORD-123**.

Th

**Note:** Verbose tool results cost tokens on every later turn. Prefer the concise payload.


In [57]:
verbose = {
    "order_id": "ORD-123",
    "status": "delivered",
    "total": 149.99,
    "_debug": {
        "query_ms": 42,
        "cache": "miss",
        "shard": "eu-west-2a",
        "raw_row": [1, 2, 3] * 40,
        "trace_id": "a1b2c3" * 10,
    },
    "internal_metadata": {
        "schema_version": 7,
        "migrated_at": "2024-01-01",
    },
}

concise = {"order_id": "ORD-123", "status": "delivered", "total": 149.99}

for name, payload in [("verbose", verbose), ("concise", concise)]:
    r = client.messages.create(
        model=MODEL,
        max_tokens=64,
        messages=[
            {
                "role": "user",
                "content": f"Summarise this order: {json.dumps(payload)}",
            }
        ],
    )
    print(f"{name:>8}: {r.usage.input_tokens} input tokens")

 verbose: 529 input tokens
 concise: 39 input tokens


**Note:** Pull transactional facts into a persistent `case_facts` block outside summarised history.


In [58]:
def case_facts(customer, order):
    return {
        "customer_id": customer["customer_id"],
        "order_id": order["order_id"],
        "amount": order["total"],
        "days_since_delivery": order["days_since_delivery"],
    }


# sent with every prompt, outside the summarised history
print(
    json.dumps(
        case_facts(CUSTOMERS["alice@example.com"], ORDERS["ORD-123"]),
        indent=2,
    )
)

{
  "customer_id": "CUST-001",
  "order_id": "ORD-123",
  "amount": 149.99,
  "days_since_delivery": 6
}


**Note:** `tool_choice` forces a JSON shape. Schema catches malformed output — not a wrong-but-valid answer.


In [ ]:
extract_tool = {
    "name": "record_ticket",
    "description": "Record a structured support ticket.",
    "input_schema": {
        "type": "object",
        "properties": {
            "intent": {
                "type": "string",
                "enum": ["refund", "delivery", "technical", "other"],
            },
            "order_id": {"type": ["string", "null"]},
            "urgency": {
                "type": "string",
                "enum": ["low", "medium", "high"],
            },
            "summary": {"type": "string"},
        },
        "required": ["intent", "order_id", "urgency", "summary"],
    },
}

r = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    tools=[extract_tool],
    tool_choice={"type": "tool", "name": "record_ticket"},
    messages=[
        {
            "role": "user",
            "content": "my monitor arrived cracked, order ORD-456, need this sorted today",
        }
    ],
)

ticket = next(b for b in r.content if b.type == "tool_use")
print(json.dumps(ticket.input, indent=2))

{
  "intent": "delivery",
  "order_id": "ORD-456",
  "urgency": "high",
  "summary": "Customer received a cracked monitor. Item was damaged upon arrival. Customer requires urgent resolution same day."
}


## Demo 5 — Hard rules in code, judgement in the prompt

**Note:** Policy you cannot afford to lose lives in code (ceiling check, prerequisite gate) — deterministic, not prompt-hope. Escalation criteria and few-shot escalate-vs-resolve examples belong in the system prompt. Ceiling rejection is a **business** error, not permission.


**Note:** Ceiling check lives in the tool (raw-API stand-in for a hook). Rejection is a **business** error.


In [60]:
REFUND_CEILING = 500.00


def process_refund_guarded(order_id, amount):
    order = ORDERS.get(order_id.upper().strip())
    if order is None:
        return tool_error("validation", False, "Unknown order.")
    if abs(amount - order["total"]) > 0.01:
        return tool_error(
            "validation",
            False,
            "Amount does not match the order total.",
            order_total=order["total"],
        )
    if amount > REFUND_CEILING:
        return tool_error(
            "business",
            False,
            f"Refunds above {REFUND_CEILING} require human approval. "
            "Call escalate_to_human.",
        )
    return process_refund(order_id, amount)


def escalate_to_human(reason, order_id=None):
    return {
        "escalated": True,
        "reason": reason,
        "order_id": order_id,
        "ticket": f"ESC-{random.randint(1000, 9999)}",
    }


# Prerequisite gate: process_refund blocked until get_customer returns verified
VERIFIED = {"customer_id": None}


def get_customer_gated(email):
    result = get_customer(email)
    if result.get("found") and result.get("verified"):
        VERIFIED["customer_id"] = result["customer_id"]
    else:
        VERIFIED["customer_id"] = None
    return result


def require_verified_customer(fn):
    def wrapper(**kwargs):
        if VERIFIED["customer_id"] is None:
            return tool_error(
                "permission",
                False,
                "Customer identity not verified. Call get_customer first.",
            )
        return fn(**kwargs)

    return wrapper


lookup_order_gated = require_verified_customer(lookup_order)
process_refund_gated = require_verified_customer(process_refund_guarded)

TOOL_FUNCTIONS["get_customer"] = get_customer_gated
TOOL_FUNCTIONS["lookup_order"] = lookup_order_gated
TOOL_FUNCTIONS["process_refund"] = process_refund_gated
TOOL_FUNCTIONS["escalate_to_human"] = escalate_to_human
TOOL_FUNCTIONS["calculate_refund"] = calculate_refund_v2

TOOLS_GUARDED = [
    get_customer_tool,
    {
        "name": "lookup_order",
        "description": (
            "Look up an order by order ID. Returns status, total, customer ID, "
            "and days since delivery."
        ),
        "input_schema": {
            "type": "object",
            "properties": {"order_id": {"type": "string"}},
            "required": ["order_id"],
        },
    },
    {
        "name": "process_refund",
        "description": (
            "Issue a refund when policy checks pass. Returns a business error if "
            "the amount exceeds the ceiling. On permission or business blocks, "
            "call escalate_to_human."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "order_id": {"type": "string"},
                "amount": {"type": "number"},
            },
            "required": ["order_id", "amount"],
        },
    },
    {
        "name": "escalate_to_human",
        "description": (
            "Escalate to a human agent when a refund cannot be completed safely, "
            "a policy exception is required, or the customer asks for a human."
        ),
        "input_schema": {
            "type": "object",
            "properties": {
                "reason": {"type": "string"},
                "order_id": {"type": ["string", "null"]},
            },
            "required": ["reason"],
        },
    },
]

SYSTEM_GUARDED = (
    "You are a customer support assistant. Use tools to help the customer. "
    "Process refunds when permitted. Escalate when a tool returns a permission "
    "or business error, or when the customer needs a human.\n\n"
    "Escalate examples:\n"
    "- Customer asks for a human -> escalate_to_human\n"
    "- Refund blocked by policy ceiling or unverified account -> escalate_to_human\n"
    "Resolve examples:\n"
    "- Verified customer, eligible order under ceiling -> process_refund\n"
    "- Order status question with a verified account -> answer from lookup_order"
)

**Note:** Prerequisite gate in application code — `process_refund` / `lookup_order` blocked until `get_customer` verifies. Not a prompt instruction.


In [61]:
# Gate fires: order number, no email
VERIFIED["customer_id"] = None
print("direct gate check:", lookup_order_gated(order_id="ORD-123"))

SYSTEM_GATE = (
    "You are a customer support assistant. Use the tools available. "
    "Act on the information given; call tools rather than asking for missing fields."
)

# Force a tool call so the application gate is exercised (not a clarifying question)
messages = [{"role": "user", "content": "Refund ORD-123, my name is Alice."}]
response = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    system=SYSTEM_GATE,
    tools=TOOLS_GUARDED,
    tool_choice={"type": "any"},
    messages=messages,
)
print("stop_reason:", response.stop_reason)
messages.append({"role": "assistant", "content": response.content})

results = []
for block in response.content:
    if block.type != "tool_use":
        continue
    out = TOOL_FUNCTIONS[block.name](**block.input)
    print(f"{block.name}({block.input}) -> {out}")
    results.append(
        {"type": "tool_result", "tool_use_id": block.id, "content": json.dumps(out)}
    )
messages.append({"role": "user", "content": results})

final = client.messages.create(
    model=MODEL,
    max_tokens=1024,
    system=SYSTEM_GATE,
    tools=TOOLS_GUARDED,
    messages=messages,
)
print(final.stop_reason)
print(final.content[0].text)


direct gate check: {'ok': False, 'errorCategory': 'permission', 'isRetryable': False, 'message': 'Customer identity not verified. Call get_customer first.'}
stop_reason: tool_use
lookup_order({'order_id': 'ORD-123'}) -> {'ok': False, 'errorCategory': 'permission', 'isRetryable': False, 'message': 'Customer identity not verified. Call get_customer first.'}
end_turn
I need to verify your identity before processing a refund. Could you please provide me with the **email address** associated with your account?


**Note:** Escalation criteria + few-shot examples live in the system prompt. Bob fails verification / window / ceiling — expect `escalate_to_human`.


In [62]:
# Escalation path: unverified, outside window, above ceiling
VERIFIED["customer_id"] = None
final, transcript = run_agent(
    "I'm bob@example.com, refund ORD-456 please.",
    system=SYSTEM_GUARDED,
    tools=TOOLS_GUARDED,
)

print(final.stop_reason)
print(final.content[0].text)
print()
for m in transcript:
    if isinstance(m["content"], str):
        print(f"{m['role']:>9}: {m['content'][:70]}")
    else:
        for b in m["content"]:
            t = b.type if hasattr(b, "type") else b["type"]
            name = getattr(b, "name", None) or (
                b.get("name") if isinstance(b, dict) else None
            )
            label = f"{t}:{name}" if name else t
            print(f"{m['role']:>9}: [{label}]")


end_turn
Your case has been escalated! Here's a summary:

- **Ticket:** ESC-4038
- **Issue:** Account verification required before the refund for ORD-456 can be processed.

A human agent will reach out to you shortly to verify your identity and complete the refund. Is there anything else I can help you with?

     user: I'm bob@example.com, refund ORD-456 please.
assistant: [text]
assistant: [tool_use:get_customer]
assistant: [tool_use:lookup_order]
     user: [tool_result]
     user: [tool_result]
assistant: [text]
assistant: [tool_use:escalate_to_human]
     user: [tool_result]
